# Prerequistes

- In order to download ALL card scans we'll setup mtg-bulk-database with postgres and then update ./.env with credentials:

https://github.com/JakeTurner616/mtg-bulk-database

In [ ]:
# Cell 1: Install required libraries (if not already installed)
%pip install numpy opencv-python h5py faiss-cpu matplotlib
%pip install psycopg2-binary python-dotenv aiohttp tqdm nest_asyncio

## Descriptor Extraction Pipeline

This workflow extracts SIFT descriptors from MTG card images and stores them very efficiently:

- Uses only `.h5`, `.index`, and `id_map.json` — no SQLite or extra mappings.
- Appends to `candidate_features.h5` without overwriting existing data.
- Automatically batches descriptors to temp `.npy` files for low memory usage.
- Builds a FAISS index only when new data is added or index is missing.
- Fully resumable and idempotent — reruns won't duplicate or erase data.

# Feature extraction
- Preprocess  with CLAHE adjustments to the luminance channel, and convert back to the RGB color space
- Extract features using SIFT, apply RootSIFT normalization, and Product Quantization into an Inverted File Index
- ~~Generate a index_to_card.txt for robust searching of keypoints with RANSAC~~
- Add to existing model files instead of creating new ones each time

# Inference and Test accuracy

In [ ]:
import os
import cv2
import json
import faiss
import h5py
import random
import numpy as np
import requests
import matplotlib.pyplot as plt

# ---------------------------
# CONFIG + LOAD INDEX & ID MAP
# ---------------------------
H5_FEATURES_FILE = "resources/run/candidate_features.h5"
FAISS_INDEX_FILE = "resources/run/faiss_ivf.index"
ID_MAP_FILE = "resources/run/id_map.json"

index = faiss.read_index(FAISS_INDEX_FILE)

with open(ID_MAP_FILE, "r") as f:
    id_map = json.load(f)

# ---------------------------
# DEFINE MULTIPLE VALID IDS
# ---------------------------
sample = {
    "scryfall_ids": [
        "3394cefd-a3c6-4917-8f46-234e441ecfb6",
        "710160a6-43b4-4ba7-9dcd-93e01befc66f"
    ],
    "image_url": "https://cards.scryfall.io/large/front/3/3/3394cefd-a3c6-4917-8f46-234e441ecfb6.jpg?1592487887",
    "face_index": 0
}
image_url = sample["image_url"]
ground_truth_ids = set(sample["scryfall_ids"])  # Convert to set for fast lookup

# ---------------------------
# DESCRIPTOR EXTRACTION + VISUALIZATION
# ---------------------------
def extract_query_descriptors_from_url(image_url, max_features=100, visualize=False):
    response = requests.get(image_url, timeout=10)
    image_array = np.asarray(bytearray(response.content), dtype=np.uint8)
    image = cv2.imdecode(image_array, cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Failed to decode image from URL: {image_url}")

    image = cv2.resize(image, (256, 256))
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    L, A, B = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    L_clahe = clahe.apply(L)
    lab_clahe = cv2.merge((L_clahe, A, B))
    gray = cv2.cvtColor(cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2BGR), cv2.COLOR_BGR2GRAY)

    sift = cv2.SIFT_create(nfeatures=max_features)
    keypoints, descriptors = sift.detectAndCompute(gray, None)

    if descriptors is not None and len(keypoints) > max_features:
        sorted_kp_des = sorted(zip(keypoints, descriptors), key=lambda x: -x[0].response)
        keypoints, descriptors = zip(*sorted_kp_des[:max_features])
        keypoints, descriptors = list(keypoints), np.array(descriptors)

    if descriptors is not None:
        eps = 1e-7
        descriptors = descriptors / (descriptors.sum(axis=1, keepdims=True) + eps)
        descriptors = np.sqrt(descriptors).astype(np.float32)

    if visualize and keypoints is not None:
        image_with_kp = cv2.drawKeypoints(image, keypoints, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
        plt.figure(figsize=(8, 8))
        plt.imshow(cv2.cvtColor(image_with_kp, cv2.COLOR_BGR2RGB))
        plt.title("Query Image with SIFT Keypoints")
        plt.axis("off")
        plt.show()

    return descriptors

# ---------------------------
# FAISS MATCHING
# ---------------------------
def predict_card_from_url(image_url, index, id_map, top_k=5, visualize=False):
    descriptors = extract_query_descriptors_from_url(image_url, visualize=visualize)
    if descriptors is None:
        return None, []

    D, I = index.search(descriptors, top_k)

    prediction_counts = {}
    for indices in I:
        for i in indices:
            if i < len(id_map):
                card_id = id_map[i]
                prediction_counts[card_id] = prediction_counts.get(card_id, 0) + 1

    sorted_preds = sorted(prediction_counts.items(), key=lambda x: -x[1])
    return sorted_preds[0][0] if sorted_preds else None, sorted_preds[:top_k]

# ---------------------------
# RUN INFERENCE
# ---------------------------
top_pred, top_matches = predict_card_from_url(image_url, index, id_map, top_k=10, visualize=True)

print(f"Query image URL: {image_url}")
print(f"Valid ground truth IDs: {list(ground_truth_ids)}")
print(f"Top prediction:          {top_pred}")
print("Top matches:")
for match_id, score in top_matches:
    print(f"  {match_id}: {score}")

if top_pred in ground_truth_ids:
    print("✅ Prediction is CORRECT (matched one of the valid IDs).")
else:
    print("❌ Prediction is INCORRECT (did not match any valid IDs).")


## Package the resources.zip:

In [ ]:
import zipfile
import os

# ---------------------------
# CONFIG
# ---------------------------
OUTPUT_ZIP = 'resourcesV4.zip'
FILES_TO_INCLUDE = [
    'resources/run/candidate_features.h5',
    'resources/run/faiss_ivf.index'
    'resources/run/id_map.json',
]

# ---------------------------
# ZIP PACKAGING
# ---------------------------
with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file_path in FILES_TO_INCLUDE:
        if os.path.exists(file_path):
            arcname = os.path.relpath(file_path, start='resources')
            zipf.write(file_path, arcname=os.path.join('resources', arcname))
            print(f"✔️ Added: {file_path}")
        else:
            print(f"⚠️ Skipped (not found): {file_path}")

print(f"\n📦 Packaged inference archive saved to: {OUTPUT_ZIP}")
